<a href="https://colab.research.google.com/github/Ziqi-Li/GEO4162C/blob/fall-24/notebooks/TSP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Travelling salesman problem (TSP)

This notebook demostrates how to solve the TSP with examples of real city coordinates.

TSP requires `ortools`, so you will need to install it on Colab.

In [1]:
pip install ortools

Import needed packages

In [2]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import matplotlib.pyplot as plt

Define the cities/places/locations that need to be visited.

In [3]:
# Example: Coordinates for 5 places
cities = [
    (30.3322, -81.6557),  # Jacksonville
    (28.5383, -81.3792),  # Orlando
    (27.9506, -82.4572),  # Tampa
    (30.4383, -84.2807),  # Tallahassee
    (25.7617, -80.1918)   # Miami
]

Below is a block of complicated code, but you don't need to worry about it, simply copy, paste and execute in your own notebook.

In [4]:
def create_data_model(coordinates):
    """Stores the data for the problem."""
    data = {}
    data['locations'] = coordinates
    data['num_locations'] = len(data['locations'])
    data['depot'] = 0  # Start at the first location
    return data

def distance(lat1, lon1, lat2, lon2):
    """Euclidean distance on a plane (simplified)."""
    return ((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5

def create_distance_matrix(data):
    distance_matrix = {}
    for from_node in range(data['num_locations']):
        distance_matrix[from_node] = {}
        for to_node in range(data['num_locations']):
            if from_node == to_node:
                distance_matrix[from_node][to_node] = 0
            else:
                lat1, lon1 = data['locations'][from_node]
                lat2, lon2 = data['locations'][to_node]
                distance_matrix[from_node][to_node] = distance(lat1, lon1, lat2, lon2)
    return distance_matrix

# TSP Solution using ORTools
def solve_tsp(data):
    manager = pywrapcp.RoutingIndexManager(data['num_locations'], 1, data['depot'])
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return int(distance_matrix[from_node][to_node])

    distance_matrix = create_distance_matrix(data)
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        route = []
        index = routing.Start(0)
        while not routing.IsEnd(index):
            route.append(manager.IndexToNode(index))
            index = solution.Value(routing.NextVar(index))
        route.append(manager.IndexToNode(index))
        return route
    else:
        return None



Use the `cities` coordinates to create data and then to solve the tsp using the `solve_tsp` function.

In [5]:

data = create_data_model(cities)

route = solve_tsp(data)


If there is an optimal route, it will give the order of the visit as the indicies of your locations.

In [6]:

if route:
    print("TSP Route:", route)
else:
    print("No solution found.")


TSP Route: [0, 1, 4, 2, 3, 0]


In this case, 0->1->4->2->3->0 refers to

Jacksonville(0) -> Orlando(1) -> Miami(4) -> Tampa(2) -> Tallahassee(3) -> Jacksonville(0)

So go from Jacksonville and return back to Jacksonville, and this route is the shortest possible route.


Next, we can visualize this as a web map using `folium` package.

In [7]:
import folium


# Initialize a folium map centered at a location, with a certain zoom level
m = folium.Map(location=[28.5383, -81.3792], zoom_start=6)

# Add markers for each POI
for poi in cities:
    folium.Marker([poi[0], poi[1]]).add_to(m)

# Add lines between the POIs in the simulated route order
route_coordinates = [(cities[i][0], cities[i][1]) for i in route]

folium.PolyLine(route_coordinates, color="blue", weight=2.5, opacity=1).add_to(m)


In [8]:
#display the map
m